# QuantJourney SDK - Oil Curve Regime and Instrument Selection

This notebook demonstrates a QuantJourney SDK workflow that combines WTI reference data, futures pricing, energy ETFs, COT positioning and petroleum data to classify oil curve regimes and instrument fit.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


## Market Data Helpers

In [ ]:
def price_frame(symbol: str, start: str=START, end: str=END) -> pd.DataFrame:
    payload = qj.eod.get_historical_prices(symbol=symbol, start_date=start, end_date=end)
    rows = as_rows(payload)
    if not rows and isinstance(unwrap(payload), dict):
        value = unwrap(payload)
        rows = value.get(symbol) or value.get(symbol.upper()) or []
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'No price data returned for {symbol}')
    df['date'] = pd.to_datetime(df['date'])
    for col in ['open', 'high', 'low', 'close', 'adjusted_close', 'volume']:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'adjusted_close' in df and df['adjusted_close'].notna().any():
        df['price'] = df['adjusted_close'].fillna(df['close'])
    else:
        df['price'] = df['close']
    if 'volume' not in df:
        df['volume'] = np.nan
    return df.dropna(subset=['price']).sort_values('date').set_index('date')

def price_panel(symbols: list[str], start: str=START, end: str=END) -> tuple[pd.DataFrame, pd.DataFrame]:
    prices = {}
    volumes = {}
    for symbol in symbols:
        df = price_frame(symbol, start=start, end=end)
        prices[symbol] = df['price']
        volumes[symbol] = df['volume']
    return (pd.DataFrame(prices).dropna(how='all'), pd.DataFrame(volumes).reindex(pd.DataFrame(prices).index))

def returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how='all')

def dollar_adv(prices: pd.DataFrame, volumes: pd.DataFrame, window: int=63) -> pd.DataFrame:
    return (prices * volumes).rolling(window).mean()


In [ ]:
futures_contracts = qj.eod.get_futures_contracts(exchange='COMM')
cl_front = qj.eod.get_futures_pricing(symbol='CL1', start_date='2022-01-01', end_date=END)
cl_second = qj.eod.get_futures_pricing(symbol='CL2', start_date='2022-01-01', end_date=END)
petroleum_raw = qj.eia.get_petroleum_prices()
cot_raw = qj.cftc.get_cot_summary(symbol='CL')
prices, volumes = price_panel(['USO', 'XLE', 'XOP', 'SPY'], start='2022-01-01', end=END)


In [ ]:
def close_series(name: str, payload: Any) -> pd.Series:
    frame = pd.DataFrame(as_rows(payload))
    if frame.empty:
        return pd.Series(dtype=float, name=name)
    date_col = next((col for col in frame.columns if 'date' in str(col).lower()), frame.columns[0])
    value_col = 'close' if 'close' in frame.columns else frame.select_dtypes(include='number').columns[-1]
    frame['date'] = pd.to_datetime(frame[date_col], errors='coerce')
    frame[name] = pd.to_numeric(frame[value_col], errors='coerce')
    return frame.dropna(subset=['date', name]).set_index('date')[name].sort_index()
curve = pd.concat([close_series('CL1 front month', cl_front), close_series('CL2 second month', cl_second)], axis=1).dropna(how='all')


In [ ]:
if curve.empty:
    raise RuntimeError('No oil futures curve data returned')
curve['front_second_spread'] = curve['CL1 front month'] - curve['CL2 second month']
curve['roll_yield_proxy'] = curve['front_second_spread'] / curve['CL1 front month']
regime = np.where(curve['front_second_spread'] > 0, 'backwardation', 'contango')
instrument = pd.DataFrame({'return_63d': prices.pct_change(63).iloc[-1], 'volatility_63d': returns(prices).tail(63).std() * np.sqrt(252), 'oil_beta_proxy': returns(prices).tail(252).corrwith(curve['CL1 front month'].pct_change().reindex(prices.index))})
display(pd.Series({'contracts_rows': len(as_rows(futures_contracts)), 'petroleum_rows': len(as_rows(petroleum_raw)), 'cot_rows': len(as_rows(cot_raw)), 'latest_regime': regime[-1]}))
display(instrument.sort_values('oil_beta_proxy', ascending=False))
curve[['CL1 front month', 'CL2 second month', 'front_second_spread']].tail(504).plot(title='WTI curve and front-second spread')
plt.show()


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.